# 03. 케이스-대조 파일럿 — 대동맥판막질환(I35)의 PPG 형태학

**질문**: 대동맥판막 질환은 PPG 파형에 흔적을 남기는가? 남긴다면 어떤 지표에?

**생리학적 예측**: 판막이 좁아지면 좌심실 박출이 느리고 약해진다 (*pulsus parvus et tardus*).
말초 맥파에서는 **상승이 느려지고**(CT↑, 기울기↓), **정점이 낮고 둔해진다**(폭↓).

**교란 요인 세 가지를 반드시 통제**한다.
1. **연령** — 판막질환 환자는 고령이고, 연령 자체가 파형을 크게 바꾼다
2. **심박수** — 모든 타이밍 지표에 직접 영향
3. **리듬** — 심방세동은 박동 간격을 불규칙하게 만든다 → 정상동율동만 사용

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
from ppg_fm.config import load
from ppg_fm.morph.features import extract, bandpass, segment_beats

CFG  = load()
DATA = CFG["datasets"]["mimic_ext_ppg"]["root"]
INT  = os.path.join(ROOT, "data", "interim")
REP  = os.path.join(ROOT, "reports")

C1, C2 = "#2a78d6", "#eb6834"   # case / control
INK2, SURF = "#52514e", "#fcfcfb"

## 1. 코호트 구성 — 환자 단위 1:2 연령 매칭

세그먼트 단위가 아니라 **환자 단위로** 매칭한다. 세그먼트 단위 매칭은
같은 환자가 양쪽에 들어갈 수 있어 무효다.

In [ ]:
idx = pd.read_parquet(os.path.join(INT, "seg_index.parquet"))
idx["subject"] = idx.subject.astype(str)

h    = idx[idx.hq & (idx.rhythm == "SR")]          # 고품질 + 정상동율동
case = h[h.I35]                                     # 대동맥판막 장애
pool = h[~h.cardiac]                                # 심장질환 코드 없음

cp = case.groupby("subject").agg(age=("age","first")).dropna()
pp = pool.groupby("subject").agg(age=("age","first")).dropna()

used, pairs = set(), []
for s, a in cp.age.items():
    cand = pp[(pp.age.sub(a).abs() <= 3) & (~pp.index.isin(used))]
    take = list(cand.index[:2]); used.update(take)
    pairs += [(s, "case", a)] + [(t, "ctrl", pp.age[t]) for t in take]

MT = pd.DataFrame(pairs, columns=["subject","grp","age"])
print(f"case {(MT.grp=='case').sum():4d}명  age {MT[MT.grp=='case'].age.median():.0f}")
print(f"ctrl {(MT.grp=='ctrl').sum():4d}명  age {MT[MT.grp=='ctrl'].age.median():.0f}")
print(f"연령 차이 검정 p = {mannwhitneyu(MT[MT.grp=='case'].age, MT[MT.grp=='ctrl'].age)[1]:.3f}")

## 2. 박동 단위 특징 추출

`scripts/07_extract_all.py` 또는 `scripts/06_morph_case_control.py` 로 추출한 결과를 읽는다.
(추출은 수 분 걸리므로 별도 스크립트로 분리하고, 여기서는 결과만 분석한다.)

In [ ]:
B = pd.read_csv(os.path.join(INT, "beat_features.csv"))
print(f"박동 {len(B):,}  환자 {B.subject.nunique()}")
print(f"  case {B[B.grp=='case'].subject.nunique()} / ctrl {B[B.grp=='ctrl'].subject.nunique()}")
B.head(3)

## 3. ⚠️ 분석 단위 — 이 연구에서 가장 중요한 방법론 결정

11만 박동은 **755명**에게서 나왔다. 박동을 독립 표본으로 취급하면
가성반복(pseudo-replication)이 되어 p값이 극단적으로 부풀려진다.

아래에서 같은 데이터·같은 지표로 두 방식을 직접 비교한다.

In [ ]:
# (a) 박동 단위 — 잘못된 방식
bl_case = (B[B.grp=="case"].CT / B[B.grp=="case"].IBI).dropna()
bl_ctrl = (B[B.grp=="ctrl"].CT / B[B.grp=="ctrl"].IBI).dropna()
p_beat = mannwhitneyu(bl_case, bl_ctrl)[1]

# (b) 환자 단위 — 올바른 방식
pl = B.groupby(["subject","grp"]).apply(lambda d: (d.CT/d.IBI).median()).rename("CT_IBI").reset_index()
p_pat = mannwhitneyu(pl[pl.grp=="case"].CT_IBI, pl[pl.grp=="ctrl"].CT_IBI)[1]

print(f"박동 단위 (n={len(bl_case)+len(bl_ctrl):,})  p = {p_beat:.3e}")
print(f"환자 단위 (n={len(pl):,})       p = {p_pat:.4f}")
print(f"\n차이: {abs(np.log10(p_pat/p_beat)):.0f} 자릿수")

> **결론: 이후 모든 추론은 환자 단위로 한다.**
> 박동 단위는 분포·변동성을 기술하는 데만 쓴다.

## 4. 환자 단위 요약

각 환자에 대해
- 특징별 **중앙값** (대표값)
- 특징별 **박동간 변동성** IQR/median (박동 단위 정보를 요약)
- **검출률** (c–d파, notch 등이 잡힌 박동 비율)

In [ ]:
FEATS = ["CT","LVET","CT_over_LVET","CT_over_IBI","LVET_over_IBI","dT",
         "W25","W50","W75","W50_over_IBI","RI","notch_rel_height","IPA",
         "max_slope_norm","t_max_slope_rel",
         "b_over_a","c_over_a","d_over_a","e_over_a","aging_index"]

P = (B.groupby(["subject","grp","age"])
       .agg({**{f:"median" for f in FEATS}, "IBI":"median",
             "notch_found":"mean", "apg_found":"mean"}).reset_index())
P["HR"] = 60000 / P.IBI

# 검출률 (결측이 아니라 정보로 취급)
P = P.merge(B.groupby("subject").d_over_a.apply(lambda s: s.notna().mean()).rename("cd_rate"), on="subject")
P = P.merge(B.groupby("subject").RI.apply(lambda s: s.notna().mean()).rename("ri_rate"), on="subject")

# 박동간 변동성
cv = B.groupby("subject")[FEATS].agg(lambda s: (s.quantile(.75)-s.quantile(.25))/(abs(s.median())+1e-9))
cv.columns = [c+"_cv" for c in cv.columns]
P = P.merge(cv, on="subject")

P["is_case"] = (P.grp == "case").astype(int)
P.to_csv(os.path.join(REP, "morph_patient_features.csv"), index=False)
print(P.shape)

## 5. 군 비교 — 연령·심박수 보정 + 다중검정 보정

In [ ]:
def compare(cols):
    out = []
    for c in cols:
        s = P[[c,"is_case","age","HR"]].dropna()
        if len(s) < 60 or s[c].nunique() < 5: continue
        a, b = s[s.is_case==1][c], s[s.is_case==0][c]
        if len(a) < 20 or len(b) < 20: continue
        d = (a.mean()-b.mean()) / np.sqrt((a.var()+b.var())/2 + 1e-18)
        try:  padj = smf.ols(f'Q("{c}") ~ is_case + age + HR', s).fit().pvalues["is_case"]
        except Exception: padj = np.nan
        out.append(dict(feat=c, case=a.median(), ctrl=b.median(), d=d,
                        p_raw=mannwhitneyu(a,b)[1], p_adj=padj, n=len(s)))
    return pd.DataFrame(out)

R = pd.concat([compare(FEATS),
               compare(["cd_rate","ri_rate","notch_found","apg_found"]),
               compare([c+"_cv" for c in FEATS])], ignore_index=True).dropna(subset=["p_adj"])
R["q"] = multipletests(R.p_adj, method="fdr_bh")[1]
R = R.reindex(R.d.abs().sort_values(ascending=False).index)
R.to_csv(os.path.join(REP, "feature_effect_sizes.csv"), index=False)

print(f"FDR<0.05 통과: {(R.q<0.05).sum()} / {len(R)}\n")
R.head(15).round(4)

## 6. 해석

### 6-1. 생리학적 일관성
효과 방향이 모두 *pulsus parvus et tardus* 와 일치하는지 확인한다.

| 예측 | 지표 | 기대 방향 |
|---|---|---|
| 상승이 느려짐 | `max_slope_norm` | ↓ |
| 정점이 뒤로 밀림 | `t_max_slope_rel`, `CT_over_LVET` | ↑ |
| 정점이 좁아짐 | `W75` | ↓ |

### 6-2. 가장 강한 신호는 "검출 실패"
`cd_rate`(c–d파 검출률)가 최대 효과크기를 보인다. 통상적인 파이프라인은
이를 결측 또는 저품질로 처리해 **버리는** 정보다.

In [ ]:
for f in ["max_slope_norm","t_max_slope_rel","CT_over_LVET","W75","cd_rate"]:
    r = R[R.feat==f]
    if len(r):
        r = r.iloc[0]
        print(f"  {f:18s} case {r.case:8.3f}  ctrl {r.ctrl:8.3f}   d = {r.d:+.3f}   q = {r.q:.4f}")

## 7. 적대적 검증 — `cd_rate` 는 신호품질 아티팩트인가?

c–d파가 안 잡히는 것이 **신호가 나빠서**라면 이 결과는 무의미하다.
독립적인 품질 지표 두 개로 검증한다.

1. **박동-템플릿 상관** — 박동들이 서로 얼마나 일관된가 (높을수록 깨끗)
2. **고주파 잡음비** — 10–30 Hz 성분 / 대역내 성분

그리고 품질을 보정한 뒤에도 군 차이가 남는지 본다.

In [ ]:
import wfdb
os.chdir(DATA)

sub = pd.concat([P[P.grp=="case"].sample(90, random_state=1),
                 P[P.grp=="ctrl"].sample(90, random_state=1)])
sub["subject"] = sub.subject.astype(str)
src = pd.concat([case, pool])

rows = []
for _, r in sub.iterrows():
    ss = src[src.subject == r.subject]
    if not len(ss): continue
    tc, hf = [], []
    for fp in ss.sample(min(2, len(ss)), random_state=1).folder_path:
        try:
            rec = wfdb.rdrecord(fp)
            sig = rec.p_signal[:, rec.sig_name.index("PLETH")]; fs = rec.fs
            x = bandpass(sig, fs); z = (x-x.mean())/(x.std()+1e-9)
            W = []
            for on, sp, on2 in segment_beats(z, fs):
                seg = x[on:on2]
                if np.ptp(seg) < 1e-9: continue
                W.append(np.interp(np.linspace(0,1,100), np.linspace(0,1,len(seg)),
                                   (seg-seg.min())/np.ptp(seg)))
            if len(W) >= 5:
                W = np.array(W); tmpl = W.mean(0)
                tc.append(float(np.median([np.corrcoef(w, tmpl)[0,1] for w in W])))
            hb = bandpass(sig, fs, 10, min(30, fs/2-1))
            hf.append(float(np.std(hb)/(np.std(x)+1e-9)))
        except Exception:
            pass
    if tc:
        rows.append(dict(subject=r.subject, grp=r.grp, cd_rate=r.cd_rate,
                         age=r.age, HR=r.HR, tmpl_corr=np.mean(tc),
                         hf_ratio=np.mean(hf) if hf else np.nan))

S = pd.DataFrame(rows); S["is_case"] = (S.grp=="case").astype(int)
os.chdir(os.path.join(ROOT, "notebooks"))

print("=== 품질 지표가 두 군에서 다른가? ===")
for c in ["tmpl_corr","hf_ratio"]:
    a, b = S[S.is_case==1][c].dropna(), S[S.is_case==0][c].dropna()
    d = (a.mean()-b.mean())/np.sqrt((a.var()+b.var())/2)
    print(f"  {c:11s} case {a.median():.4f}  ctrl {b.median():.4f}  d={d:+.3f}  p={mannwhitneyu(a,b)[1]:.4f}")

m0 = smf.ols("cd_rate ~ is_case + age + HR", S).fit()
m1 = smf.ols("cd_rate ~ is_case + age + HR + tmpl_corr + hf_ratio", S).fit()
print(f"\n품질 보정 전 : is_case {m0.params['is_case']:+.4f}  p={m0.pvalues['is_case']:.2e}")
print(f"품질 보정 후 : is_case {m1.params['is_case']:+.4f}  p={m1.pvalues['is_case']:.2e}")
print(f"계수 변화 {100*(m1.params['is_case']-m0.params['is_case'])/abs(m0.params['is_case']):+.1f}%")
S.to_csv(os.path.join(REP, "cd_rate_snr_check.csv"), index=False)

> **판정**: 두 군의 신호 품질은 통계적으로 동일하고(오히려 케이스가 더 깨끗),
> 품질을 보정해도 효과가 거의 움직이지 않는다 → **아티팩트가 아니다.**

## 8. 한계

| # | 한계 | 영향 |
|---|---|---|
| 1 | **I35는 협착과 폐쇄부전을 합친 3자리 코드** — 혈역학이 정반대 | 효과 **희석**. 협착 단독이면 더 클 것 |
| 2 | ICD 코드 기반 라벨 (심초음파 확진 아님) | 오분류 존재 |
| 3 | ICU 환자 — 승압제·진정·기계환기 혼재 | 외적 타당도 제한 |
| 4 | 단일 코호트, 외부검증 없음 | 재현 필요 |
| 5 | 대조군 = "심장질환 코드 없음" ≠ 정상 | 보수적 방향 |

→ 다음: `04_phenome_wide.ipynb` (174개 질환 전체 스캔)